## Chain of Responsibility

---

> **In one line.** A request travels down a linked line of handlers $H_1 \rightarrow H_2 \rightarrow \cdots \rightarrow H_n$; the first one whose predicate fires handles it and the line stops, and the sender knows only the head $H_1$.

### 1. The request and the chain

Let $r$ be the **request** — the object set in motion: a support ticket, an HTTP request, an expense claim. It does not know its own destination. Standing ready to receive it is a finite ordered list of **handlers** $H_1, H_2, \dots, H_n$, each a single link. The defining structural fact is that the links are *threaded*: every $H_i$ holds a reference to its successor $H_{i+1}$, so the whole forms a one-directional sequence

$$\text{Chain} \;=\; H_1 \rightarrow H_2 \rightarrow \cdots \rightarrow H_n,$$

where each arrow $H_i \rightarrow H_{i+1}$ reads "passes to". The sender deposits $r$ at the head $H_1$ and nothing more; it never names a recipient.

### 2. The pass-or-handle decision

Each handler $H_i$ carries two pieces of equipment. The first is a **predicate** $p_i : r \mapsto \{\top, \bot\}$ — the boolean test *"can $H_i$ handle this request?"* (e.g. severity $\leq$ low). The second is an **action** $f_i(r)$ — what $H_i$ actually *does* once it accepts. When $r$ arrives at $H_i$, exactly one of two things happens:

$$H_i(r) \;=\; \begin{cases} f_i(r) & \text{if } p_i(r) = \top \quad(\text{accept: handle and stop})\\[2pt] H_{i+1}(r) & \text{if } p_i(r) = \bot \quad(\text{decline: pass on})\end{cases}$$

Here $\top$ ("true") means $H_i$ accepts and handles $r$; $\bot$ ("false", no match) means $H_i$ forwards $r$ untouched to its successor. Unfolding this recursion down the whole chain collapses it to a single guarded dispatch: $\text{handle}(r)$ returns the action of the **first** handler whose predicate holds.

$$\boxed{\;\text{handle}(r) \;=\; \begin{cases} f_1(r) & \text{if } p_1(r) = \top \\ f_2(r) & \text{else if } p_2(r) = \top \\ \;\;\vdots & \\ f_n(r) & \text{else if } p_n(r) = \top \\ \bot & \text{if no } p_i(r) = \top \end{cases}\;}$$

The final line is the escape hatch: if every predicate returns $\bot$, no handler claims $r$ and the request goes **unhandled**.

### 3. The data flow

A request walks the chain link by link, each $\bot$ advancing it one step, until a $\top$ absorbs it:

$$\text{sender} \xrightarrow{\;r\;} H_1 \xrightarrow{\;\bot\;} H_2 \xrightarrow{\;\bot\;} \cdots \xrightarrow{\;\bot\;} H_k \xrightarrow{\;\top\;} f_k(r)\;\blacksquare$$

The arrow into $H_1$ is all the sender ever sees; everything to its right is hidden machinery. The terminal $\blacksquare$ marks early termination — handlers $H_{k+1}, \dots, H_n$ are never reached.

### 4. Conditions

1. **Sender decoupling** — the sender's only contract is to hand $r$ to the head. It is blind to which link resolves it:
   $$\text{sender} \;\xrightarrow{\;r\;}\; H_1, \qquad \text{sender} \perp \{H_2, \dots, H_n\}.$$
2. **Early termination** — the first $\top$ is final. If $p_k(r) = \top$ then $f_k$ runs and the recursion halts; for every $j > k$, $H_j$ never sees $r$:
   $$p_k(r) = \top \;\Longrightarrow\; H_{k+1}(r), \dots, H_n(r) \text{ are never evaluated.}$$
3. **Extensibility** — inserting a new handler $H_{n+1}$ at any position only re-wires the local $\rightarrow$ references. No existing $H_i$ — neither its $p_i$ nor its $f_i$ — is touched:
   $$H_1 \rightarrow \cdots \rightarrow H_{n+1} \rightarrow \cdots \rightarrow H_n, \qquad \forall\, i:\; (p_i, f_i) \text{ unchanged}.$$

&nbsp;

> 🎫 Expense approval. A £50 claim ($r$) lands on the team lead ($H_1$). Too large, so $p_1(r) = \bot$ and it passes to the manager ($H_2$). Now $p_2(r) = \top$, so $f_2(r)$ approves it and the chain stops. The employee handed the claim to one desk and never knew who would sign it off.

### Exercise 13 — Support Ticket System

---

**Scenario:** Tickets have severity: low, medium, high, critical. $H_1$ = JuniorSupport ($p_1$: severity = low), $H_2$ = SeniorSupport ($p_2$: medium), $H_3$ = Manager ($p_3$: high), $H_4$ = Director ($p_4$: critical).

**Your task:** Build the chain $H_1 \rightarrow H_2 \rightarrow H_3 \rightarrow H_4$. A ticket is automatically routed to the first $H_i$ where $p_i(r) = \top$.

```python
chain = JuniorSupport(SeniorSupport(Manager(Director())))
chain.handle(Ticket("Bug", severity="low"))        # p_1 = ⊤, f_1 handles
chain.handle(Ticket("Outage", severity="critical")) # passes to H_4
```

**Hints**

- Each $H_i$ stores `self._next = H_{i+1}`. If $p_i(r) = \bot$, call `self._next.handle(r)`. Build a `BaseHandler` with a default `handle()` that passes to next if set.
- Add a new severity level — you add one $H_{n+1}$ and insert it. No existing handler changes. This is the extensibility condition.

In [ ]:
# --------------------------------
# The request r

class Ticket:
    def __init__(self, title, severity):
        self.title = title
        self.severity = severity            # 'low' | 'medium' | 'high' | 'critical'

# --------------------------------
# BaseHandler — holds reference to next H_{i+1}; default handle() passes along

class BaseHandler:
    def __init__(self, next_handler=None):
        self._next = next_handler            # reference to H_{i+1} (or None)

    def handle(self, r):
        # if p_i(r) = ⊤ -> f_i(r) (subclass decides). else -> pass to H_{i+1}
        if self._can_handle(r):              # p_i(r)
            self._act(r)                     # f_i(r): handle, chain stops
        elif self._next is not None:
            self._next.handle(r)             # p_i(r) = ⊥ -> pass to H_{i+1}
        else:
            print(f"Unhandled: '{r.title}' ({r.severity})")   # no p_i(r) = ⊤

    def _can_handle(self, r):                # p_i(r) — override per handler
        ...

    def _act(self, r):                       # f_i(r) — override per handler
        ...

# --------------------------------
# H_1: JuniorSupport — p_1: severity == 'low'

class JuniorSupport(BaseHandler):
    def _can_handle(self, r):                # p_1(r)
        ...
    def _act(self, r):                       # f_1(r)
        ...                                  # e.g. print(f"Junior handles '{r.title}'")

# --------------------------------
# H_2: SeniorSupport — p_2: severity == 'medium'

class SeniorSupport(BaseHandler):
    def _can_handle(self, r):                # p_2(r)
        ...
    def _act(self, r):                       # f_2(r)
        ...

# --------------------------------
# H_3: Manager — p_3: severity == 'high'

class Manager(BaseHandler):
    def _can_handle(self, r):                # p_3(r)
        ...
    def _act(self, r):                       # f_3(r)
        ...

# --------------------------------
# H_4: Director — p_4: severity == 'critical'

class Director(BaseHandler):
    def _can_handle(self, r):                # p_4(r)
        ...
    def _act(self, r):                       # f_4(r)
        ...

# --------------------------------
# Build the chain H_1 -> H_2 -> H_3 -> H_4
chain = JuniorSupport(SeniorSupport(Manager(Director())))
chain.handle(Ticket("Typo on page", severity="low"))        # p_1 = ⊤
chain.handle(Ticket("Slow query", severity="medium"))       # passes to H_2
chain.handle(Ticket("Login broken", severity="high"))       # passes to H_3
chain.handle(Ticket("Full outage", severity="critical"))    # passes to H_4

### Exercise 14 — HTTP Middleware Pipeline

---

**Scenario:** An HTTP request $r$ passes through: auth check ($H_1$) → rate limiter ($H_2$) → logger ($H_3$) → actual handler ($H_4$). Each $H_i$ can either pass $r$ to $H_{i+1}$ or short-circuit with a response.

**Your task:** Build a middleware chain. Each link receives $(r, \text{next})$ — calling `next(r)` passes to $H_{i+1}$; returning early short-circuits.

```python
pipeline = Pipeline([auth, rate_limit, logger, final_handler])
pipeline.run({"path": "/data", "token": "valid"})   # passes all the way to H_4
pipeline.run({"path": "/data", "token": None})      # H_1 short-circuits -> 401
```

**Hints**

- Short-circuit is the Chain pattern with early termination: auth failure sets $p_1(r) = \top$ but $f_1(r) = \text{"401 Unauthorized"}$ — $H_2, H_3, H_4$ never see the request.
- Each middleware is a function `(r, next) -> response`. Call `next(r)` to delegate to $H_{i+1}$; `return` a response directly to short-circuit.

In [ ]:
# --------------------------------
# Pipeline — wires H_1 -> H_2 -> ... -> H_n so each link calls the next

class Pipeline:
    def __init__(self, middlewares):
        self._middlewares = middlewares       # [H_1, H_2, ..., H_n]

    def run(self, r):
        # build the chain back-to-front so each H_i receives (r, next)
        def make_next(i):
            if i >= len(self._middlewares):
                return lambda req: "404 No Handler"     # no p_i(r) = ⊤
            return lambda req: self._middlewares[i](req, make_next(i + 1))
        return make_next(0)(r)

# --------------------------------
# H_1: auth — p_1: token missing -> short-circuit f_1(r) = '401'

def auth(r, nxt):
    # if not r.get('token'): return '401 Unauthorized'  (early termination)
    # else: return nxt(r)  (pass to H_2)
    ...

# --------------------------------
# H_2: rate limiter — p_2: over limit -> short-circuit '429'

def rate_limit(r, nxt):
    # if over limit: return '429 Too Many Requests'
    # else: return nxt(r)  (pass to H_3)
    ...

# --------------------------------
# H_3: logger — never short-circuits; logs then delegates to H_4

def logger(r, nxt):
    # print(f"LOG: {r['path']}"); return nxt(r)
    ...

# --------------------------------
# H_4: final handler — the actual handler; always produces a response

def final_handler(r, nxt):
    # return f"200 OK: served {r['path']}"
    ...

# --------------------------------
pipeline = Pipeline([auth, rate_limit, logger, final_handler])
print(pipeline.run({"path": "/data", "token": "valid"}))   # -> reaches H_4
print("--------")
print(pipeline.run({"path": "/data", "token": None}))      # H_1 short-circuits